# Lecture 08: Unsupervised Learning: Clustering — Exercises

Companion to the note [Noter små](../Noter%20små.md).

Practise k-means the way the exam asks (iterations and SSE by hand), why the centroid is the mean, the effect of initialisation, outliers and feature scales (min-max and z-score normalisation), supervised and unsupervised evaluation, and soft clustering with EM. Then implement k-means, the elbow method, restarts and a 1-D EM from scratch.

**16 exercises** · 🧠 Concept ×3 · ✍️ By hand ×6 · 💻 Code ×7

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from scipy.stats import zscore, norm
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics.cluster import contingency_matrix
from sklearn.datasets import make_blobs, load_iris, load_digits
rng = np.random.default_rng(8)
# Exam-style 2-D points for k-means by hand
X_hand = np.array([(1, 7), (2, 6), (2, 7), (3, 5), (5, 4), (6, 4), (6, 5), (7, 4)], float)
C_init = np.array([(2, 6), (3, 5)], float)
# Points on very different scales (for normalisation)
D_scale = np.array([(5, 3), (48, 4), (66, 12), (80, 10), (95, 9)], float)

## Part A · Concepts

### Exercise 1 · What makes a good clustering, and how do we know?
*🧠 Concept · ★☆☆*

1. In terms of distances, what does Lecture 08 aim for in a good clustering?
2. The note warns that a clustering algorithm "will return a clustering, even if there is no meaningful structure in the data". Give an example.
3. Explain the difference between supervised and unsupervised evaluation of a clustering, and one weakness of each.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Think of within-cluster and between-cluster distances, and of running 3-means on uniformly random points.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. Small **within-cluster** distances (compact clusters) and large **between-cluster** distances (well separated clusters).
2. Run 3-means on points drawn uniformly from a square: you get three tidy-looking regions, although there are no groups at all.
3. *Supervised* evaluation compares clusters with known labels (e.g. iris species). Weakness: if you had labels you would usually do
   supervised learning, and the labels are not necessarily the most natural grouping. *Unsupervised* evaluation uses only the data and the
   clustering (e.g. SSE, distances between clusters). Weakness: such measures always improve with more clusters, so they cannot alone tell
   which clusters are "real".

</details>

### Exercise 2 · Convergence of k-means
*🧠 Concept · ★★☆*

1. Argue that neither step of k-means (assignment, centroid update) can increase the SSE.
2. Why does that guarantee termination?
3. Why is the result not necessarily the global optimum, and what does the lecture recommend?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

For termination: how many different assignments of $n$ points to $k$ clusters are there?

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. *Assignment*: each point moves to its closest centroid, so its squared distance cannot grow. *Update*: for a fixed cluster, the mean
   minimises the sum of squared distances (next exercise), so replacing the old centroid by the mean cannot increase the SSE.
2. The SSE is non-increasing and there are only finitely many ($\le k^n$) assignments. An assignment cannot repeat after a strict decrease, so
   after finitely many steps the assignment (and hence the centroids) stops changing.
3. k-means is a local search: it stops in the first local optimum it reaches, which depends on the initial centroids. Remedy:
   **several random restarts**, keeping the result with the lowest SSE (and smarter seeding such as k-means++).

</details>

### Exercise 3 · Outliers and initialisation
*🧠 Concept · ★★☆*

One-dimensional data: $\{1, 2, 3,\ 10, 11, 12,\ 40\}$ with $k = 2$.

1. Run 2-means from initial centres $c = (5, 40)$. Where do the centres end and what is the SSE?
2. Now start from $c = (2, 11)$, which looks like it separates the two natural groups. Trace the iterations. Where do you end?
3. What does this say about k-means and outliers, and what preprocessing does the lecture suggest?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Simulate the algorithm: assign each point to the closer centre, average, repeat until nothing changes. Watch the point 10 in part 2.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. $\{1,2,3,10,11,12\}$ → $6.5$ and $\{40\}$ → $40$; stable. SSE $= 2(5.5^2 + 4.5^2 + 3.5^2) = 125.5$.
2. Iteration 1: $\{1,2,3\}$ → 2, $\{10,11,12,40\}$ → 18.25. The outlier drags the second centre to 18.25, so now 10 is closer to 2
   ($8 < 8.25$). Iteration 2: $\{1,2,3,10\}$ → 4, $\{11,12,40\}$ → 21. Then 11 moves ($7 < 10$): centres 5.4 and 26; then 12 moves
   ($6.6 < 14$). It ends in **the same clustering as part 1**: the outlier alone, all ordinary points together.
3. Because squared errors are dominated by the outlier, the SSE-optimal 2-clustering spends a whole cluster on it, and the natural two-group
   structure is lost. The lecture recommends **outlier detection and elimination** before clustering (or using more clusters).

</details>

## Part B · By hand

### Exercise 4 · Two iterations of k-means
*✍️ By hand · ★☆☆*

Cluster the eight points $(1,7), (2,6), (2,7), (3,5), (5,4), (6,4), (6,5), (7,4)$ with $k = 2$, Euclidean distance and initial centres
$c_1 = (2,6)$, $c_2 = (3,5)$. Perform two iterations (assignment + update) and give the centres after each iteration.

In [ ]:
C_iter1 = None   # [[x, y], [x, y]]
C_iter2 = None

check('centres after iteration 1', C_iter1, KMeans(2, init=C_init, n_init=1, max_iter=1).fit(X_hand).cluster_centers_, tol=1e-3)
check('centres after iteration 2', C_iter2, KMeans(2, init=C_init, n_init=1, max_iter=2).fit(X_hand).cluster_centers_, tol=1e-3)

<details>
<summary><b>💡 Hint 1</b></summary>

Iteration 1: $(3,5)$ is a centre itself; $(5,4)$, $(6,4)$, $(6,5)$, $(7,4)$ are clearly closer to $c_2$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Iteration 2: compare $(3,5)$ with the new centres $(1.67, 6.67)$ and $(5.4, 4.4)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

**Iteration 1.** Cluster 1: $(1,7), (2,6), (2,7)$; cluster 2: $(3,5), (5,4), (6,4), (6,5), (7,4)$.
New centres: $c_1 = (5/3, 20/3) \approx (1.67, 6.67)$, $c_2 = (27/5, 22/5) = (5.4, 4.4)$.

**Iteration 2.** $(3,5)$: squared distance to $c_1$ is $1.78 + 2.78 = 4.56$, to $c_2$ it is $5.76 + 0.36 = 6.12$, so it moves to cluster 1.
New centres: $c_1 = (2, 6.25)$, $c_2 = (6, 4.25)$. A third iteration changes nothing: converged.

```python
C_iter1 = [[5/3, 20/3], [5.4, 4.4]]
C_iter2 = [[2, 6.25], [6, 4.25]]
```

</details>

### Exercise 5 · SSE before and after
*✍️ By hand · ★☆☆*

For the same data, compute the SSE $\sum_i\sum_{x\in C_i} d(c_i, x)^2$

1. with the **initial** centres (and the points assigned to their closest initial centre), and
2. with the centres after iteration 2 (and their clusters).

In [ ]:
sse_init = None
sse_final = None

def _sse(C):
    l = np.argmin(((X_hand[:, None] - C[None]) ** 2).sum(2), 1)
    return sum(((X_hand[l == k] - C[k]) ** 2).sum() for k in range(len(C)))
check('SSE initial', sse_init, _sse(C_init)); check('SSE final', sse_final, _sse(np.array([[2, 6.25], [6, 4.25]])))

<details>
<summary><b>💡 Hint</b></summary>

Initial: $(1,7), (2,6), (2,7)$ belong to $(2,6)$; the other five to $(3,5)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. Cluster of $(2,6)$: $2 + 0 + 1 = 3$. Cluster of $(3,5)$: $0 + 5 + 10 + 9 + 17 = 41$. **SSE = 44.**
2. $c_1 = (2, 6.25)$: $(1,7)$: $1 + 0.5625$; $(2,6)$: $0.0625$; $(2,7)$: $0.5625$; $(3,5)$: $1 + 1.5625$ → $4.75$.
   $c_2 = (6, 4.25)$: $1.0625 + 0.0625 + 0.5625 + 1.0625 = 2.75$. **SSE = 7.5.**

```python
sse_init = 44.0
sse_final = 7.5
```

</details>

### Exercise 6 · The mean minimises the SSE
*✍️ By hand · ★★☆*

Show that for a fixed cluster $C_i$ the centroid $c$ minimising $\sum_{x\in C_i}\lVert x - c\rVert^2$ is the mean $\frac{1}{|C_i|}\sum_{x\in C_i}x$.
Which point minimises $\sum_{x\in C_i}\lVert x - c\rVert_1$ instead (one dimension is enough)?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Set the gradient with respect to $c$ to zero. For the $L_1$ case think about how the sum changes when $c$ moves past a data point.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\nabla_c\sum_x\lVert x-c\rVert^2 = -2\sum_x(x - c) = -2\big(\sum_x x - |C_i|c\big) = 0 \Rightarrow c = \frac{1}{|C_i|}\sum_x x$.
The function is convex (a sum of convex quadratics), so this stationary point is the global minimum.

For the absolute loss in 1-D the derivative is $\#\{x < c\} - \#\{x > c\}$, which is zero at the **median**. That gives *k-medians*,
which is more robust to outliers (compare the mean/median remark for 0–1/L1/L2 losses in Lecture 05).

</details>

### Exercise 7 · Scales change who is nearest
*✍️ By hand · ★★☆*

Five points: $d_1 = (5, 3)$, $d_2 = (48, 4)$, $d_3 = (66, 12)$, $d_4 = (80, 10)$, $d_5 = (95, 9)$.

1. Which point is closest to $d_2$ in Euclidean distance?
2. Z-score normalise each attribute, $A_i \mapsto \frac{A_i - \text{mean}(A_i)}{\text{std}(A_i)}$ (population standard deviation, divide by $n$).
   Give the normalised $d_2$ and the point closest to $d_2$ now.

In [ ]:
nearest_raw = None   # e.g. 'd3'
z_d2 = None          # [z1, z2]
nearest_z = None

_names = ['d1', 'd2', 'd3', 'd4', 'd5']
_dr = np.linalg.norm(D_scale - D_scale[1], axis=1); _dr[1] = np.inf
_Z = zscore(D_scale); _dz = np.linalg.norm(_Z - _Z[1], axis=1); _dz[1] = np.inf
check('nearest (raw)', nearest_raw, _names[int(np.argmin(_dr))]); check('z-scored d2', z_d2, _Z[1], tol=1e-3)
check('nearest (z-score)', nearest_z, _names[int(np.argmin(_dz))])

<details>
<summary><b>💡 Hint 1</b></summary>

Raw distances are dominated by the first attribute (range 90 vs range 9).

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Means: $(58.8, 7.6)$; standard deviations: $\approx(31.06, 3.50)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. Raw distances to $d_2$: $d_1$: 43.0, $d_3$: 19.7, $d_4$: 32.6, $d_5$: 47.3. Nearest: **$d_3$**.
2. $z(d_2) = \big(\frac{48-58.8}{31.06}, \frac{4-7.6}{3.50}\big) \approx (-0.348, -1.029)$. In z-space the distances are $d_1$: 1.41, $d_3$: 2.36,
   $d_4$: 2.00, $d_5$: 2.08, so the nearest is now **$d_1$**: on the second attribute $d_1$ and $d_2$ are almost identical, which the raw
   distance ignored.

```python
nearest_raw = 'd3'
z_d2 = [-0.348, -1.029]
nearest_z = 'd1'
```

</details>

### Exercise 8 · Min-max normalisation
*✍️ By hand · ★☆☆*

Min-max normalise the second attribute of the five points above, $A_i \mapsto \frac{A_i - \min A_i}{\max A_i - \min A_i}$.
Then add a sixth point with second attribute 120 and normalise again. What happens to the original five values, and why does the lecture call
min-max normalisation "very sensitive to outliers"?

In [ ]:
mm = None          # 5 values without the outlier
mm_outlier = None  # the same 5 points after adding the value 120

_a = D_scale[:, 1]; check('min-max', mm, (_a - _a.min()) / (_a.max() - _a.min()), tol=1e-3)
_b = np.r_[_a, 120]; check('with outlier', mm_outlier, ((_b - _b.min()) / (_b.max() - _b.min()))[:5], tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

Without the outlier min = 3 and max = 12; with it max = 120.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Without the outlier: $(0, 0.111, 1, 0.778, 0.667)$. With the value 120: $(0, 0.0085, 0.077, 0.060, 0.051)$.
A single extreme value squeezes all ordinary points into a tiny interval, so that attribute almost stops mattering in distances.
Z-score normalisation is affected too, but less, and outliers are better removed first.

```python
mm = [0, 1/9, 1, 7/9, 6/9]
mm_outlier = [0, 1/117, 9/117, 7/117, 6/117]
```

</details>

### Exercise 9 · One EM iteration for a 1-D mixture
*✍️ By hand · ★★★*

Soft clustering with two Gaussian components with fixed standard deviation $\sigma = 1$, current means $\mu_1 = 2$, $\mu_2 = 5$ and mixing weights
$\pi_1 = \pi_2 = 0.5$. Data: $x = (1, 2, 6, 7)$.

1. **E-step:** compute the responsibilities $r_{i1} = \frac{\pi_1\mathcal N(x_i;\mu_1,1)}{\pi_1\mathcal N(x_i;\mu_1,1) + \pi_2\mathcal N(x_i;\mu_2,1)}$.
2. **M-step:** update $\mu_1 = \frac{\sum_i r_{i1}x_i}{\sum_i r_{i1}}$, $\mu_2$ likewise with $r_{i2} = 1 - r_{i1}$, and $\pi_1 = \frac1n\sum_i r_{i1}$.

In [ ]:
resp1 = None   # 4 responsibilities for component 1
mu1_new = None
mu2_new = None
pi1_new = None

_x = np.array([1, 2, 6, 7.]); _a = 0.5 * norm.pdf(_x, 2, 1); _b = 0.5 * norm.pdf(_x, 5, 1); _r = _a / (_a + _b)
check('responsibilities', resp1, _r, tol=1e-3)
check('mu1', mu1_new, (_r * _x).sum() / _r.sum(), tol=1e-3); check('mu2', mu2_new, ((1 - _r) * _x).sum() / (1 - _r).sum(), tol=1e-3)
check('pi1', pi1_new, _r.mean(), tol=1e-3)

<details>
<summary><b>💡 Hint 1</b></summary>

With equal $\pi$ and $\sigma$, $r_{i1} = \frac{1}{1 + \exp\big(\frac{(x_i-\mu_1)^2 - (x_i-\mu_2)^2}{2}\big)}$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

For $x = 1$: $\frac{(1-2)^2 - (1-5)^2}{2} = \frac{1 - 16}{2} = -7.5$, so $r = 1/(1 + e^{-7.5}) \approx 0.9994$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Exponents $\frac{(x-2)^2 - (x-5)^2}{2}$: $-7.5,\ -4.5,\ 7.5,\ 10.5$. Responsibilities
$r_{\cdot1} \approx (0.99945,\ 0.98901,\ 0.00055,\ 0.00003)$.

$\sum r_{i1} = 1.9890$; $\mu_1 = (0.99945 + 1.97803 + 0.00332 + 0.00019)/1.9890 \approx 1.499$.
$\sum r_{i2} = 2.0110$; $\mu_2 \approx 6.474$. $\pi_1 \approx 0.497$.

Unlike k-means, every point contributes to every component with a weight. Here the weights are almost 0/1, so EM behaves like k-means.

```python
resp1 = [0.99945, 0.98901, 0.00055, 0.0000275]
mu1_new = 1.4987
mu2_new = 6.4740
pi1_new = 0.4973
```

</details>

## Part C · Code

### Exercise 10 · k-means from scratch
*💻 Code · ★★☆*

Implement `kmeans(X, C0, max_iter=100)` (Lloyd's algorithm exactly as in the note: assign to the closest centroid, recompute the means, stop
when the centroids do not change). Return `(centroids, labels, sse)`. Test it on blobs and compare with `KMeans(init=C0, n_init=1)`.

In [ ]:
X_blobs, y_blobs = make_blobs(n_samples=300, centers=4, cluster_std=1.0, random_state=3)
C0_blobs = X_blobs[[0, 1, 2, 3]]

def kmeans(X, C0, max_iter=100):
    # TODO
    return None

res_km = kmeans(X_blobs, C0_blobs)

_ref = KMeans(4, init=C0_blobs, n_init=1, tol=0).fit(X_blobs)
check('final SSE matches sklearn inertia', None if res_km is None else res_km[2], _ref.inertia_, tol=1e-6)
check('centroids match sklearn', None if res_km is None else res_km[0], _ref.cluster_centers_, tol=1e-6)

<details>
<summary><b>💡 Hint</b></summary>

Distances to all centroids at once: `((X[:, None, :] - C[None, :, :]) ** 2).sum(2)`. Keep the old centroid if a cluster becomes empty.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The loop usually converges in a handful of iterations. Your SSE equals sklearn's `inertia_`, the within-cluster sum of squared errors.

```python
X_blobs, y_blobs = make_blobs(n_samples=300, centers=4, cluster_std=1.0, random_state=3)
C0_blobs = X_blobs[[0, 1, 2, 3]]

def kmeans(X, C0, max_iter=100):
    C = np.array(C0, float)
    for _ in range(max_iter):
        labels = np.argmin(((X[:, None, :] - C[None, :, :]) ** 2).sum(2), axis=1)
        C_new = np.array([X[labels == k].mean(0) if np.any(labels == k) else C[k] for k in range(len(C))])
        if np.allclose(C_new, C):
            break
        C = C_new
    labels = np.argmin(((X[:, None, :] - C[None, :, :]) ** 2).sum(2), axis=1)
    sse = float(sum(((X[labels == k] - C[k]) ** 2).sum() for k in range(len(C))))
    return C, labels, sse

res_km = kmeans(X_blobs, C0_blobs)
print(res_km[0], res_km[2])
```

</details>

### Exercise 11 · Random restarts
*💻 Code · ★★☆*

Run your `kmeans` 20 times on `X_blobs` with $k = 4$, each time initialised with 4 random data points (`np.random.default_rng(seed)` for
`seed = 0..19`). Store the 20 SSE values in `restart_sses` and the best one in `best_sse`. Are all runs equally good?

In [ ]:
restart_sses = None
best_sse = None

if restart_sses is not None:
    check('20 runs', len(restart_sses), 20)
    check('best SSE is the minimum', best_sse, min(restart_sses))
    check('best restart is as good as sklearn (n_init=20)', best_sse <= KMeans(4, n_init=20, random_state=0).fit(X_blobs).inertia_ + 1e-6, True)
else:
    check('20 runs', None, 20)

<details>
<summary><b>💡 Hint</b></summary>

`idx = np.random.default_rng(seed).choice(len(X_blobs), 4, replace=False)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Several runs land in a worse local optimum (e.g. two centroids sharing one blob while another centroid covers two blobs), with a clearly
higher SSE. Keeping the best of many restarts is cheap insurance, and it is what `n_init` does in sklearn.

```python
restart_sses = []
for seed in range(20):
    idx = np.random.default_rng(seed).choice(len(X_blobs), 4, replace=False)
    restart_sses.append(kmeans(X_blobs, X_blobs[idx])[2])
best_sse = min(restart_sses)
print(np.round(restart_sses, 1))
```

</details>

### Exercise 12 · The elbow method
*💻 Code · ★★☆*

For $k = 1,\dots,8$ compute the best-of-10-restarts SSE (use `KMeans(k, n_init=10, random_state=0)`) for

* `X_blobs` (4 real clusters), and
* `X_unif`: 300 points uniform in the bounding box of `X_blobs`.

Store the lists in `sse_blobs` and `sse_unif`. How can these curves be used to pick $k$? Store your choice for the blobs in `k_elbow`.

In [ ]:
X_unif = rng.uniform(X_blobs.min(0), X_blobs.max(0), size=(300, 2))
sse_blobs = None
sse_unif = None
k_elbow = None

if sse_blobs is not None:
    check('SSE decreases with k', bool(np.all(np.diff(sse_blobs) <= 1e-9)), True)
    check('elbow at the true number of clusters', k_elbow, 4)
else:
    check('SSE decreases with k', None, True)

<details>
<summary><b>💡 Hint</b></summary>

Look at the relative drop `sse[k-1] / sse[k]`: it is large up to the true $k$ and small afterwards.

</details>

<details>
<summary><b>✅ Solution</b></summary>

SSE always decreases with $k$ (with $k = n$ it is 0), so we cannot simply minimise it. For the blobs the curve drops steeply until $k = 4$ and
then flattens: the **elbow**. For uniform data there is no elbow, just a smooth decrease, a hint that there are no real clusters.

```python
X_unif = rng.uniform(X_blobs.min(0), X_blobs.max(0), size=(300, 2))
sse_blobs = [KMeans(k, n_init=10, random_state=0).fit(X_blobs).inertia_ for k in range(1, 9)]
sse_unif = [KMeans(k, n_init=10, random_state=0).fit(X_unif).inertia_ for k in range(1, 9)]
drops = np.array(sse_blobs[:-1]) / np.array(sse_blobs[1:])
print(np.round(drops, 2))
# the elbow is the last k with a big relative drop before the curve flattens
k_elbow = int(np.max(np.where(drops > 1.5)[0])) + 2
```

</details>

### Exercise 13 · Supervised evaluation: purity
*💻 Code · ★★☆*

Cluster iris (all 4 features) with `KMeans(3, n_init=10, random_state=0)`. Implement `purity(labels_true, labels_pred)`: for each cluster
take the count of its most frequent true class, sum, divide by $n$. Also compute the adjusted Rand index with sklearn.
Why does purity alone reward many small clusters?

In [ ]:
X_ir, y_ir = load_iris(return_X_y=True)
labels_ir = KMeans(3, n_init=10, random_state=0).fit_predict(X_ir)

def purity(labels_true, labels_pred):
    # TODO
    return None

pur = purity(y_ir, labels_ir)
ari = None

_cm = contingency_matrix(y_ir, labels_ir)
check('purity', pur, _cm.max(0).sum() / _cm.sum())
check('ARI', ari, adjusted_rand_score(y_ir, labels_ir))
check('purity of singletons is 1', None if pur is None else purity(y_ir, np.arange(len(y_ir))), 1.0)

<details>
<summary><b>💡 Hint</b></summary>

Loop over the cluster ids and use `np.bincount(labels_true[labels_pred == c]).max()`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Purity is about 0.89: setosa is found perfectly, versicolor and virginica overlap. Putting every point in its own cluster gives purity 1,
so purity must be read together with the number of clusters; ARI corrects for chance and penalises such trivial solutions (ARI ≈ 0.73 here).

```python
X_ir, y_ir = load_iris(return_X_y=True)
labels_ir = KMeans(3, n_init=10, random_state=0).fit_predict(X_ir)

def purity(labels_true, labels_pred):
    labels_true, labels_pred = np.asarray(labels_true), np.asarray(labels_pred)
    total = sum(np.bincount(labels_true[labels_pred == c]).max() for c in np.unique(labels_pred))
    return total / len(labels_true)

pur = purity(y_ir, labels_ir)
ari = adjusted_rand_score(y_ir, labels_ir)
print(pur, ari)
```

</details>

### Exercise 14 · Normalisation before clustering
*💻 Code · ★★☆*

Simulate 200 people from two groups that differ in *height* (cm), while *income* (DKK) is pure noise with a huge spread:
the code is given. Cluster with 2-means on the raw data and on z-scored data and compare the ARI with the true groups
(`ari_raw`, `ari_z`). Explain the result.

In [ ]:
group = np.repeat([0, 1], 100)
height = np.where(group == 0, 165, 185) + rng.normal(0, 5, 200)
income = rng.normal(400_000, 80_000, 200)
X_people = np.c_[height, income]
ari_raw = None
ari_z = None

if ari_raw is not None:
    check('z-scoring recovers the groups', ari_z > 0.8, True)
    check('raw clustering does not', ari_raw < 0.2, True)
else:
    check('z-scoring recovers the groups', None, True)

<details>
<summary><b>💡 Hint</b></summary>

`zscore(X_people)` standardises each column. Use `KMeans(2, n_init=10, random_state=0)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

On raw data the Euclidean distance is dominated by income (spread ~80 000 vs ~10 cm), so k-means splits by income, which is noise (ARI ≈ 0).
After z-scoring both attributes have unit variance, the height difference becomes visible and the groups are recovered (ARI ≈ 0.86).
This is the "different measuring scales" issue from the lecture.

```python
group = np.repeat([0, 1], 100)
height = np.where(group == 0, 165, 185) + rng.normal(0, 5, 200)
income = rng.normal(400_000, 80_000, 200)
X_people = np.c_[height, income]
ari_raw = adjusted_rand_score(group, KMeans(2, n_init=10, random_state=0).fit_predict(X_people))
ari_z = adjusted_rand_score(group, KMeans(2, n_init=10, random_state=0).fit_predict(zscore(X_people)))
print(ari_raw, ari_z)
```

</details>

### Exercise 15 · EM for a 1-D Gaussian mixture
*💻 Code · ★★★*

Implement `em_gmm_1d(x, mu, sigma, pi, n_iter)` for $K$ components (arrays of length $K$), updating means, standard deviations and weights.
Fit 2 components to the data below starting from `mu = [0, 1]`, `sigma = [1, 1]`, `pi = [0.5, 0.5]` and compare with
`GaussianMixture(2)`.

In [ ]:
x_mix = np.r_[rng.normal(-2, 0.7, 300), rng.normal(3, 1.2, 200)]

def em_gmm_1d(x, mu, sigma, pi, n_iter=200):
    # TODO: return (mu, sigma, pi) as arrays
    return None

em_res = em_gmm_1d(x_mix, np.array([0., 1.]), np.array([1., 1.]), np.array([0.5, 0.5]))

_g = GaussianMixture(2, random_state=0, tol=1e-8, max_iter=1000).fit(x_mix[:, None])
_o = np.argsort(_g.means_.ravel())
if em_res is None:
    check('means match GaussianMixture', None, 0)
else:
    _oo = np.argsort(em_res[0])
    check('means match GaussianMixture', np.asarray(em_res[0])[_oo], _g.means_.ravel()[_o], tol=1e-3)
    check('weights match GaussianMixture', np.asarray(em_res[2])[_oo], _g.weights_[_o], tol=1e-3)

<details>
<summary><b>💡 Hint 1</b></summary>

E-step: `R = pi * norm.pdf(x[:, None], mu, sigma)`, then normalise each row.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

M-step: `Nk = R.sum(0)`, `mu = R.T @ x / Nk`, `sigma = sqrt((R * (x[:, None] - mu) ** 2).sum(0) / Nk)`, `pi = Nk / len(x)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

EM recovers means near $-2$ and $3$ and weights near $0.6/0.4$. Each iteration cannot decrease the likelihood (analogous to k-means never
increasing the SSE), and like k-means it can get stuck in local optima, so restarts help.

```python
x_mix = np.r_[rng.normal(-2, 0.7, 300), rng.normal(3, 1.2, 200)]

def em_gmm_1d(x, mu, sigma, pi, n_iter=200):
    mu, sigma, pi = (np.array(a, float) for a in (mu, sigma, pi))
    for _ in range(n_iter):
        R = pi * norm.pdf(x[:, None], mu, sigma)
        R /= R.sum(1, keepdims=True)
        Nk = R.sum(0)
        mu = R.T @ x / Nk
        sigma = np.sqrt((R * (x[:, None] - mu) ** 2).sum(0) / Nk)
        pi = Nk / len(x)
    return mu, sigma, pi

em_res = em_gmm_1d(x_mix, np.array([0., 1.]), np.array([1., 1.]), np.array([0.5, 0.5]))
print(em_res)
```

</details>

### Exercise 16 · Semi-supervised learning: cluster, then label
*💻 Code · ★★★*

Labelling is expensive. On the digits data (`load_digits`), pretend you may only look at the labels of **5 random images per cluster**.
Cluster all images with `KMeans(30, n_init=4, random_state=0)`, label each cluster by the majority of its 5 inspected images
(`np.random.default_rng(0)` for the choice), and assign that label to every image in the cluster. Store the resulting accuracy in `ssl_acc`.
How many labels did you use?

In [ ]:
X_dig, y_dig = load_digits(return_X_y=True)
ssl_acc = None

check('cluster-then-label accuracy > 0.75', None if ssl_acc is None else ssl_acc > 0.75, True)

<details>
<summary><b>💡 Hint</b></summary>

For cluster `c`: `idx = np.where(labels == c)[0]`, `seen = r.choice(idx, min(5, len(idx)), replace=False)`, `np.bincount(y_dig[seen]).argmax()`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

With only $30\times5 = 150$ labels (8 % of 1797 images) the cluster-then-label approach reaches roughly 85–90 % accuracy, because
k-means groups images of the same digit. This is the idea behind semi-supervised learning from the end of the lecture: unsupervised structure
plus a few labels.

```python
X_dig, y_dig = load_digits(return_X_y=True)
labels_d = KMeans(30, n_init=4, random_state=0).fit_predict(X_dig)
r = np.random.default_rng(0)
pred = np.empty_like(y_dig)
for c in range(30):
    idx = np.where(labels_d == c)[0]
    seen = r.choice(idx, min(5, len(idx)), replace=False)
    pred[idx] = np.bincount(y_dig[seen]).argmax()
ssl_acc = np.mean(pred == y_dig)
print(ssl_acc)
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Noter små](../Noter%20små.md).*